In [1]:
import torch
print("GPU Available:" , torch.cuda.is_available())


GPU Available: True


In [2]:
# 1. Install necessary libraries
!pip install -U sentence-transformers pandas

# 2. Import them
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

# 3. Connect the model to the GPU (Cuda)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2').to(device)

print(f"Model is running on: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 117.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.0
    Uninstalling sentence-transformers-5.4.0:
      Successfully uninstalled sentence-transformers-5.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model is running on: cuda


In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load dataset
df = pd.read_csv("training_data.csv").dropna()

questions = df.iloc[:, 0].tolist() # First column: Questions
answers = df.iloc[:, 1].tolist()   # Second column: Answers

# Encode corpus
print("Encoding data...")
corpus_embeddings = model.encode(questions, convert_to_tensor=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding data...


In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load dataset
df = pd.read_csv("training_data.csv").dropna()
# 2. Prepare the Knowledge Base
questions = df.iloc[:, 0].tolist() # First column: Questions
answers = df.iloc[:, 1].tolist()   # Second column: Answers

# 3. Encode the questions using the GPU
# 'corpus_embeddings' will now live on your GPU for lightning-fast matching
print("Encoding your tourism data...")
corpus_embeddings = model.encode(questions, convert_to_tensor=True)

# 4. Define the Search Function
def get_answer(user_query):
    # Move query to GPU and encode
    query_embedding = model.encode(user_query, convert_to_tensor=True)

    # Use semantic search to find the best match
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=1)

    best_hit = hits[0][0]
    score = best_hit['score']
    index = best_hit['corpus_id']

    if score > 0.65:
        return f"Bot: {answers[index]} (Confidence: {score:.2f})"
    else:
        return "Bot: Gaaffii keessan sirriitti hin hubanne. Maaloo irra deebi'aa."

# 5. Test it!
print("-" * 30)
test_q = "Lalibela eessatti argama?"
print(f"User: {test_q}")
print(get_answer(test_q))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding your tourism data...
------------------------------
User: Lalibela eessatti argama?
Bot: Oromiyaa keessatti argama (Confidence: 0.80)


In [10]:
from sentence_transformers import util

# 1. Load data
df_train = pd.read_csv('training_data.csv')
train_questions = df_train.iloc[:, 0].tolist()
train_answers = df_train.iloc[:, 1].tolist()

df_test = pd.read_csv('testing_data.csv')
test_questions = df_test.iloc[:, 0].tolist()
test_answers = df_test.iloc[:, 1].tolist()

# 2. Encode training corpus
corpus_embeddings = model.encode(train_questions, convert_to_tensor=True)

# 3. Precompute test answer embeddings (FAST)
test_answer_embeddings = model.encode(test_answers, convert_to_tensor=True)

correct_count = 0
threshold = 0.75
total_test_queries = len(test_questions)

# 4. Evaluation loop
for i in range(total_test_queries):
    query = test_questions[i]
    expected_emb = test_answer_embeddings[i]

    query_emb = model.encode(query, convert_to_tensor=True)

    hits = util.semantic_search(query_emb, corpus_embeddings, top_k=1)
    idx = hits[0][0]['corpus_id']

    predicted = train_answers[idx]

    predicted_emb = model.encode(predicted, convert_to_tensor=True)

    sim = util.cos_sim(predicted_emb, expected_emb).item()

    if sim > threshold:
        correct_count += 1
    else:
        print("❌ Query:", query)
        print("Expected:", test_answers[i])
        print("Predicted:", predicted)
        print("Similarity:", sim)
        print()

# 5. Final score
accuracy = (correct_count / total_test_queries) * 100

print("-" * 30)
print(f"FINAL ACCURACY: {accuracy:.2f}%")

❌ Query: Wanchi keessatti maal gochuu danda’ama?

Expected: Farda yaabbachuu, bishaan irra deemuu fi hiking gochuu danda’ama
Predicted: Daakaa bishaanii fi bashannana
Similarity: 0.5178595781326294

❌ Query: Debre Libanos eessa keessatti argama?
Expected: Debre Libanos Oromiyaa keessatti argama

Predicted: Addis Ababa keessatti argama
Similarity: 0.5780237913131714

❌ Query: Debre Libanos maalin beekama?
Expected: Gadaa kiristaanaa fi bifa uumamaa gaariidhaan beekama

Predicted: Tulluu fi bosona qaba
Similarity: 0.5211138129234314

❌ Query: Debre Libanos keessatti maal daawwatamuu danda’a?
Expected: Gadaa fi laga Abbay ilaaluun danda’ama
Predicted: Zebra fi bineensota
Similarity: 0.20520249009132385

❌ Query: Abbay Gorge eessa keessatti argama?
Expected: Abbay Gorge Oromiyaa fi Amaaraa gidduutti argama

Predicted: Wanchi Oromiyaa keessatti naannoo Shawaa Kibbaa keessatti argama
Similarity: 0.6185614466667175

❌ Query: Abbay Gorge maalin beekama?

Expected: Bakka gadi fagoo fi bifa lafa

In [12]:
!git config --global user.email "baaghilabdullah@gmail.com"
!git config --global user.name "abudi"

In [13]:


!git init


!git add .

!git commit -m "Initial commit: Tourism QA System for Afaan Oromo"

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
[master (root-commit) 0276351] Initial commit: Tourism QA System for Afaan Oromo
 23 files changed, 51237 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_config

In [14]:
!git branch -M main


In [15]:
!git rm -r --cached .config/
!git rm -r --cached sample_data/
!git commit --amend -m "Initial commit: Tourism QA System for Afaan Oromo (Cleaned)"


rm '.config/.last_opt_in_prompt.yaml'
rm '.config/.last_survey_prompt.yaml'
rm '.config/.last_update_check.json'
rm '.config/active_config'
rm '.config/config_sentinel'
rm '.config/configurations/config_default'
rm '.config/default_configs.db'
rm '.config/gce'
rm '.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db'
rm '.config/logs/2026.04.16/13.32.51.059837.log'
rm '.config/logs/2026.04.16/13.33.15.290817.log'
rm '.config/logs/2026.04.16/13.33.27.800089.log'
rm '.config/logs/2026.04.16/13.33.29.579401.log'
rm '.config/logs/2026.04.16/13.33.41.236186.log'
rm '.config/logs/2026.04.16/13.33.42.096053.log'
rm 'sample_data/README.md'
rm 'sample_data/anscombe.json'
rm 'sample_data/california_housing_test.csv'
rm 'sample_data/california_housing_train.csv'
rm 'sample_data/mnist_test.csv'
rm 'sample_data/mnist_train_small.csv'
[main 99f7800] Initial commit: Tourism QA System for Afaan Oromo (Cleaned)
 Date: Wed Apr 22 21:03:18 2026 +0000
 2 files changed, 178 insertions(+)
